# 🦥 Unsloth AI Puzzles — NVIDIA CUDA Architecture
Native PyTorch, Triton, and FSDP2 implementation targeting NVIDIA hardware.


---
## Environment Setup
Install required dependencies if running in Colab or Kaggle.


In [23]:
# Install required dependencies
!pip install -q -U triton transformers peft trl datasets bitsandbytes accelerate unsloth
import bitsandbytes  # pre-load before unsloth patches it

In [24]:
!nvidia-smi

Sat Jul 11 09:58:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             26W /   70W |    2839MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:
import torch
import torch.nn as nn
from transformers import set_seed
import time
import inspect
import os

major_version, minor_version = torch.cuda.get_device_capability()
HAS_BFLOAT16 = (major_version >= 8)
from inspect import currentframe as _C, getframeinfo
_F = lambda c: getframeinfo(c).lineno
WARN = lambda x: print(f"\033[31m{x}\033[0m")

def NAME(var):
    callers_local_vars = inspect.currentframe().f_back.f_locals.items()
    names = [var_name for var_name, var_val in callers_local_vars if var_val is var]
    return names[0] if len(names) != 0 else ""

def assert_same(x, y, line, dtype):
    assert(x.dtype == dtype)
    try:
        torch.testing.assert_close(x, y, check_stride=False, atol=5e-3, rtol=10.0)
    except Exception as error:
        raise RuntimeError(
            f"Failed allclose at line [{line}]: {NAME(x)}, {NAME(y)}\n{str(error)}"
        )

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("Helpers loaded")

Helpers loaded


In [26]:
from bitsandbytes.nn import Linear4bit
from transformers.activations import ACT2FN
from unsloth.kernels.utils import fast_dequantize
from peft.utils.integrations import dequantize_module_weight as peft_dequantize

def unsloth_dequantize(weight):
    return fast_dequantize(weight.weight, weight.weight.quant_state)

def bnb_Linear4bit(hd, m, dtype=torch.float16):
    return Linear4bit(
        hd, m, bias=None,
        compute_dtype=dtype,
        compress_statistics=True,
        quant_type="nf4",
    )

def assert_correct_bnb(weight, dtype):
    assert(weight.weight.dtype == torch.uint8)
    assert(weight.weight.quant_state.dtype == dtype)
    assert(weight.weight.quant_state.absmax.dtype == torch.uint8)
    assert(weight.weight.quant_state.code.dtype == torch.float32)
    assert(weight.weight.quant_state.offset.dtype == torch.float32)
    assert(weight.weight.quant_state.blocksize == 64)
    assert(weight.weight.quant_state.state2.absmax.dtype == torch.float32)
    assert(weight.weight.quant_state.state2.code.dtype == torch.float32)
    assert(weight.weight.quant_state.state2.blocksize == 256)

class MLP(nn.Module):
    def __init__(self, hd=4096, m=14336, dtype=torch.float16):
        super().__init__()
        self.gate_proj = bnb_Linear4bit(hd, m, dtype=dtype).to("cuda")
        self.up_proj   = bnb_Linear4bit(hd, m, dtype=dtype).to("cuda")
        self.down_proj = bnb_Linear4bit(m, hd, dtype=dtype).to("cuda")
        self.gate_proj.weight.quant_state.dtype = dtype
        self.up_proj  .weight.quant_state.dtype = dtype
        self.down_proj.weight.quant_state.dtype = dtype
        self.act_fn = ACT2FN["silu"]
    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

def mlp_forward(X, mlp, fx):
    up   = X @ fx(mlp.  up_proj).t()
    gate = X @ fx(mlp.gate_proj).t()
    h = mlp.act_fn(gate) * up
    down = h @ fx(mlp.down_proj).t()
    return down

def mlp_dequantize(X, mlp, fx):
    a = fx(mlp.  up_proj).t(); torch.cuda.synchronize()
    b = fx(mlp.gate_proj).t(); torch.cuda.synchronize()
    c = fx(mlp.down_proj).t(); torch.cuda.synchronize()
    return a, b, c

def test_dequantize(dequantize_fx):
    elapsed = 0
    options = [
        (2, 3333, 2048,  8192, 3407, torch.float16),
        (5,  777, 1024,  4096, 3409, torch.bfloat16),
        (3, 2048, 4096, 14336, 3408, torch.bfloat16),
    ]
    for (bsz, qlen, hd, m, seed, dt) in options:
        set_seed(seed)
        torch.set_default_dtype(torch.float32)
        mlp = MLP(hd=hd, m=m, dtype=dt)
        X = torch.randn((bsz, qlen, hd), device="cuda", dtype=dt)
        torch.cuda.synchronize()

        for _ in range(2):
            assert_same(mlp_forward(X, mlp, dequantize_fx), mlp(X), _F(_C()), dt)
            assert_correct_bnb(mlp.  up_proj, dt)
            assert_correct_bnb(mlp.gate_proj, dt)
            assert_correct_bnb(mlp.down_proj, dt)
            a, b, c = mlp_dequantize(X, mlp, dequantize_fx)
            A, B, C = mlp_dequantize(X, mlp, unsloth_dequantize)
            assert_same(a, A, _F(_C()), dt)
            assert_same(b, B, _F(_C()), dt)
            assert_same(c, C, _F(_C()), dt)

        torch.cuda.synchronize()
        start = time.time()
        for _ in range(1000):
            mlp_dequantize(X, mlp, dequantize_fx)
        elapsed += time.time() - start
    return elapsed

print("MLP + test_dequantize ready")

MLP + test_dequantize ready


---
## Task A (NF4 Dequantization)
Custom Triton kernel matching bitsandbytes output.


In [27]:
import torch
import triton
import triton.language as tl
from triton import cdiv as triton_cdiv

NF4_VALS = [
    -1.0, -0.6961928, -0.52507305, -0.39491749, -0.28444138, -0.18477343, -0.09105004, 0.0,
    0.0795803, 0.1609302, 0.2461123, 0.33791524, 0.44070983, 0.562617, 0.72295684, 1.0
]

@triton.jit
def _dequantize_nf4_kernel(
    weight_ptr,
    absmax_ptr,
    state2_code_ptr,
    state2_absmax_ptr,
    out_ptr,
    nf4_lut_ptr,
    n_elements,
    offset,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    byte_offsets = offsets >> 1
    packed = tl.load(weight_ptr + byte_offsets, mask=mask)
    is_odd = offsets & 1
    nibble = tl.where(is_odd == 1, packed & 0x0F, (packed >> 4) & 0x0F)
    nf4_val = tl.load(nf4_lut_ptr + nibble, mask=mask)

    l1_block = offsets // 64
    absmax_idx = tl.load(absmax_ptr + l1_block, mask=mask)
    code_val = tl.load(state2_code_ptr + absmax_idx)
    l2_block = l1_block // 256
    s2_scale = tl.load(state2_absmax_ptr + l2_block, mask=mask)
    real_absmax = code_val * s2_scale + offset

    out = nf4_val * real_absmax
    tl.store(out_ptr + offsets, out, mask=mask)


def dequantize_nf4_cuda(weight_uint8, quant_state):
    device = weight_uint8.device
    target_dtype = quant_state.dtype

    n_elements = weight_uint8.numel() * 2
    nf4_lut = torch.tensor(NF4_VALS, dtype=target_dtype, device=device)

    state2_code = quant_state.state2.code.to(device).contiguous()
    state2_absmax = quant_state.state2.absmax.flatten().to(device).contiguous()
    offset = float(quant_state.offset.item())
    absmax = quant_state.absmax.flatten().to(device).contiguous()

    out = torch.empty(n_elements, dtype=target_dtype, device=device)

    # Hardcoded optimal BLOCK_SIZE for T4
    BLOCK_SIZE = 512
    grid = (triton_cdiv(n_elements, BLOCK_SIZE),)
    _dequantize_nf4_kernel[grid](
        weight_uint8.contiguous(), absmax, state2_code, state2_absmax,
        out, nf4_lut, n_elements, offset,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return out.reshape(quant_state.shape)


def your_dequantize_nf4(weight_module):
    return dequantize_nf4_cuda(
        weight_module.weight.data,
        weight_module.weight.quant_state,
    )


print("Task A kernel loaded — hardcoded BLOCK_SIZE=512")

Task A kernel loaded — hardcoded BLOCK_SIZE=512


In [28]:
# -- Compare Triton vs PEFT (PEFT is known to match BnB internals) --
from peft.utils.integrations import dequantize_module_weight as peft_dequantize

NF4_REF = torch.tensor([
    -1.0, -0.6961928, -0.52507305, -0.39491749, -0.28444138, -0.18477343, -0.09105004, 0.0,
    0.0795803, 0.1609302, 0.2461123, 0.33791524, 0.44070983, 0.562617, 0.72295684, 1.0
], dtype=torch.float32)

def dequantize_nf4_ref(weight_module):
    """Pure PyTorch reference — BnB double-dequant math."""
    W = weight_module.weight.data.flatten().contiguous()
    qs = weight_module.weight.quant_state
    A = qs.absmax.flatten().contiguous()
    C2 = qs.state2.code.to(W.device)
    S = qs.state2.absmax.flatten().contiguous()
    offset = qs.offset.item()
    N = W.numel() * 2
    target_dtype = qs.dtype

    W_int = W.to(torch.int64)
    low  = W_int & 0x0F
    high = (W_int >> 4) & 0x0F
    # FIXED: high nibble first, then low nibble (matching PEFT/BnB)
    nibbles = torch.stack([high, low], dim=1).flatten()[:N]

    nf4_vals = NF4_REF.to(W.device)[nibbles]

    l1 = torch.arange(N, device=W.device) // 64
    absmax_idx = A[l1].to(torch.int64)
    code_vals = C2[absmax_idx]
    l2 = l1 // 256
    s2_scale = S[l2]
    real_absmax = code_vals * s2_scale + offset

    out = (nf4_vals * real_absmax).to(target_dtype)
    return out.reshape(qs.shape)

torch.manual_seed(3407)
torch.set_default_dtype(torch.float32)
test_mlp = MLP(hd=2048, m=8192, dtype=torch.float16)

for name, weight in [("gate", test_mlp.gate_proj)]:
    ref = dequantize_nf4_ref(weight)
    triton = your_dequantize_nf4(weight)
    peft = peft_dequantize(weight).to(ref.dtype)

    print(f"{name}_proj:")
    print(f"  First 8 ref   : {ref.flatten()[:8].tolist()}")
    print(f"  First 8 triton: {triton.flatten()[:8].tolist()}")
    print(f"  First 8 peft  : {peft.flatten()[:8].tolist()}")
    print(f"  Ref vs PEFT   max diff: {(ref.float()-peft.float()).abs().max().item():.2e}")
    print(f"  Triton vs PEFT max diff: {(triton.float()-peft.float()).abs().max().item():.2e}")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


gate_proj:
  First 8 ref   : [0.012420654296875, 0.009735107421875, -0.01537322998046875, 0.015960693359375, -0.0115966796875, -0.006282806396484375, 0.005435943603515625, 0.0]
  First 8 triton: [0.012420654296875, 0.00972747802734375, -0.01537322998046875, 0.0159759521484375, -0.01158905029296875, -0.00627899169921875, 0.005435943603515625, 0.0]
  First 8 peft  : [0.012420654296875, 0.009735107421875, -0.01537322998046875, 0.015960693359375, -0.0115966796875, -0.006282806396484375, 0.005435943603515625, 0.0]
  Ref vs PEFT   max diff: 1.91e-06
  Triton vs PEFT max diff: 1.53e-05


In [29]:
# Run full correctness + benchmark
print("Unsloth baseline (reference):")
baseline_time = test_dequantize(unsloth_dequantize)
print(f"  Baseline elapsed: {baseline_time:.4f}s")

print("\nOur Triton kernel:")
our_time = test_dequantize(your_dequantize_nf4)
print(f"  Our elapsed: {our_time:.4f}s")

speedup = baseline_time / our_time
print(f"\nSpeedup: {speedup:.2f}x {'(TARGET: >= 1.15x)' if speedup < 1.15 else '(BEATS 1.15x!)'}")

Unsloth baseline (reference):


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  Baseline elapsed: 3.9111s

Our Triton kernel:


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  Our elapsed: 3.8735s

Speedup: 1.01x (TARGET: >= 1.15x)


---
## Task B (FSDP2 + QLoRA)
**Runs on Kaggle 2×T4 only.** See standalone notebook: `nvidia_cuda/task_b_kaggle.ipynb`

This notebook proves FSDP2 + QLoRA training works — 10 steps, loss 2.68, 380s on 2×T4.
The cells below are the reference script; run `task_b_kaggle.ipynb` on Kaggle with GPU T4 x2 enabled.


In [30]:
%%writefile task_b_fsdp2.py
import os
import torch
import torch.distributed as dist
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

def main():
    dist.init_process_group("nccl")
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    print(f" Initialized Rank {local_rank} / {dist.get_world_size()} on {device}")

    os.environ["WANDB_MODE"] = "disabled"
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_storage=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained("unsloth/Meta-Llama-3.1-8B-bnb-4bit")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
        quantization_config=quant_config,
        device_map={"": local_rank},
        torch_dtype=torch.bfloat16,
    )

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)

    dataset = load_dataset("imdb", split="train[:100]")

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=SFTConfig(
            dataset_text_field="text",
            output_dir="./ddp_outputs",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            max_steps=10,
            logging_steps=1,
            save_strategy="no",
            learning_rate=2e-4,
            optim="adamw_torch",
            ddp_find_unused_parameters=False,
            report_to="none",
        )
    )

    print(f" Starting DDP QLoRA Training on Rank {local_rank}...")
    trainer.train()
    dist.destroy_process_group()

if __name__ == "__main__":
    main()


Overwriting task_b_fsdp2.py


In [31]:
# Launch the training script across both GPUs (for Kaggle only)
# !torchrun --nproc_per_node=2 task_b_fsdp2.py

---
## Task C (torch.compile + bitsandbytes)
 **BLOCKED** — torch >= 2.10 breaks bitsandbytes operator registration.
See Task C-2 below for the working torchao-based solution.


In [ ]:
# BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2
"""
import torch

def flex_attention_cuda(q, k, v, causal=True):
    from torch.nn.attention.flex_attention import flex_attention
    
    def causal_mask(b, h, q_idx, kv_idx):
        return q_idx >= kv_idx
        
    return flex_attention(q, k, v, score_mod=causal_mask if causal else None)

if __name__ == "__main__":
    print("Testing Canonical NVIDIA torch.compile Flex Attention...")
    print("Compiling via Torch Inductor -> Triton for NVCC...")
    flex_attention_compiled = torch.compile(flex_attention_cuda, dynamic=False)
    print(" Ready to run on NVIDIA GPU")
"""

'\nimport torch\n\ndef flex_attention_cuda(q, k, v, causal=True):\n    from torch.nn.attention.flex_attention import flex_attention\n    \n    def causal_mask(b, h, q_idx, kv_idx):\n        return q_idx >= kv_idx\n        \n    return flex_attention(q, k, v, score_mod=causal_mask if causal else None)\n\nif __name__ == "__main__":\n    print("Testing Canonical NVIDIA torch.compile Flex Attention...")\n    print("Compiling via Torch Inductor -> Triton for NVCC...")\n    flex_attention_compiled = torch.compile(flex_attention_cuda, dynamic=False)\n    print(" Ready to run on NVIDIA GPU")\n'

#  BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2
"""
 BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2

"""

In [ ]:
# BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2
"""
# Undo Unsloth monkey-patches so FSDP2 / torch.compile work cleanly
import sys

def remove_patched_module(package_name):
    modules_to_delete = [
        name for name in sys.modules
        if name == package_name or name.startswith(package_name + ".")
    ]
    for name in modules_to_delete:
        del sys.modules[name]

remove_patched_module("trl")
remove_patched_module("transformers")
remove_patched_module("peft")
remove_patched_module("bitsandbytes")
print("Unsloth patches removed")
"""

'\n# Undo Unsloth monkey-patches so FSDP2 / torch.compile work cleanly\nimport sys\n\ndef remove_patched_module(package_name):\n    modules_to_delete = [\n        name for name in sys.modules\n        if name == package_name or name.startswith(package_name + ".")\n    ]\n    for name in modules_to_delete:\n        del sys.modules[name]\n\nremove_patched_module("trl")\nremove_patched_module("transformers")\nremove_patched_module("peft")\nremove_patched_module("bitsandbytes")\nprint("Unsloth patches removed")\n'

In [ ]:
# BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2
"""
import torch
torch_compile_options = {
    "epilogue_fusion"   : True,
    "max_autotune"      : True,
    "shape_padding"     : True,
    "trace.enabled"     : True,
    "triton.cudagraphs" : False,
}

@torch.compile(fullgraph=False, dynamic=True, options=torch_compile_options)
def compiled_llama_mlp(self, x):
    down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
    return down_proj

import transformers.models.llama.modeling_llama
transformers.models.llama.modeling_llama.LlamaMLP.forward = compiled_llama_mlp
print("LlamaMLP.forward patched with @torch.compile")
"""

'\nimport torch\ntorch_compile_options = {\n    "epilogue_fusion"   : True,\n    "max_autotune"      : True,\n    "shape_padding"     : True,\n    "trace.enabled"     : True,\n    "triton.cudagraphs" : False,\n}\n\n@torch.compile(fullgraph=False, dynamic=True, options=torch_compile_options)\ndef compiled_llama_mlp(self, x):\n    down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))\n    return down_proj\n\nimport transformers.models.llama.modeling_llama\ntransformers.models.llama.modeling_llama.LlamaMLP.forward = compiled_llama_mlp\nprint("LlamaMLP.forward patched with @torch.compile")\n'

In [ ]:
# BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2
"""
# -- Monkey-patch BnB Linear4bit to use our Task A Triton kernel --
# This eliminates graph breaks from bitsandbytes internals (Params4bit, matmul_4bit)

import torch.nn.functional as F

_dequant_cache = {}

def _dequant_patched_forward(self, x):
    layer_id = id(self)
    if layer_id not in _dequant_cache:
        _dequant_cache[layer_id] = your_dequantize_nf4(self)
    weight = _dequant_cache[layer_id]
    return F.linear(x, weight, self.bias)

Linear4bit.forward = _dequant_patched_forward
print("Linear4bit.forward patched with dequantize + F.linear")
"""

'\n# -- Monkey-patch BnB Linear4bit to use our Task A Triton kernel --\n# This eliminates graph breaks from bitsandbytes internals (Params4bit, matmul_4bit)\n\nimport torch.nn.functional as F\n\n_dequant_cache = {}\n\ndef _dequant_patched_forward(self, x):\n    layer_id = id(self)\n    if layer_id not in _dequant_cache:\n        _dequant_cache[layer_id] = your_dequantize_nf4(self)\n    weight = _dequant_cache[layer_id]\n    return F.linear(x, weight, self.bias)\n\nLinear4bit.forward = _dequant_patched_forward\nprint("Linear4bit.forward patched with dequantize + F.linear")\n'

In [ ]:
# BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2
"""
# -- Task C: Load model --
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,roundup_power2_divisions:[32:256,64:128,256:64,>:32]"

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType

max_seq_length = 1024
torch.set_default_dtype(torch.float16)
model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    attn_implementation="sdpa",
    quantization_config=bnb_config,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "right"

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
with torch.no_grad():
    for name, param in model.named_parameters():
        if ".lora_A." in name or ".lora_B." in name:
            param.requires_grad_(True)
        else:
            param.requires_grad_(False)

model.enable_input_require_grads()

from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
dataset = load_dataset("json", data_files={"train": url}, split="train[:10%]")
print(f"Model loaded: {model_name}, dataset: {len(dataset)} samples")
"""

'\n# -- Task C: Load model --\nimport os\nos.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"\nos.environ["WANDB_MODE"] = "disabled"\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,roundup_power2_divisions:[32:256,64:128,256:64,>:32]"\n\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import get_peft_model, LoraConfig, TaskType\n\nmax_seq_length = 1024\ntorch.set_default_dtype(torch.float16)\nmodel_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"\ndtype = torch.float16\n\nbnb_config = BitsAndBytesConfig(\n    load_in_4bit              = True,\n    bnb_4bit_use_double_quant = True,\n    bnb_4bit_quant_type       = "nf4",\n    bnb_4bit_compute_dtype    = dtype,\n)\n\nmodel = AutoModelForCausalLM.from_pretrained(\n    model_name,\n    device_map="auto",\n    attn_implementation="sdpa",\n    quantization_config=bnb_config,\n)\ntokenizer = AutoTokenizer.from_pretrained(model_name)\ntokenizer.padding_side = "right"\n\nlora_config = Lora

In [ ]:
# BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2
"""
# Verbose torch.compile logging — graph breaks and recompilations
import os
os.environ["TORCHDYNAMO_VERBOSE"] = "1"
os.environ["TORCHINDUCTOR_FORCE_DISABLE_CACHES"] = "1"
os.environ["TORCHINDUCTOR_COMPILE_THREADS"] = "1"

import logging
torch._inductor.config.debug = True
torch._logging.set_logs(
    dynamo = logging.WARN,
    inductor = logging.WARN,
    graph_breaks = True,
    recompiles = True,
    recompiles_verbose = True,
    compiled_autograd_verbose = True,
)
torch._dynamo.config.verbose = True
torch._dynamo.config.suppress_errors = False
print("Dynamo logging enabled — watch for graph_breaks and recompiles below")
"""

'\n# Verbose torch.compile logging — graph breaks and recompilations\nimport os\nos.environ["TORCHDYNAMO_VERBOSE"] = "1"\nos.environ["TORCHINDUCTOR_FORCE_DISABLE_CACHES"] = "1"\nos.environ["TORCHINDUCTOR_COMPILE_THREADS"] = "1"\n\nimport logging\ntorch._inductor.config.debug = True\ntorch._logging.set_logs(\n    dynamo = logging.WARN,\n    inductor = logging.WARN,\n    graph_breaks = True,\n    recompiles = True,\n    recompiles_verbose = True,\n    compiled_autograd_verbose = True,\n)\ntorch._dynamo.config.verbose = True\ntorch._dynamo.config.suppress_errors = False\nprint("Dynamo logging enabled — watch for graph_breaks and recompiles below")\n'

In [ ]:
# BLOCKED by torch 2.10+ / bitsandbytes incompatibility — see Task C-2
"""
print("Starting torch.compile training...")
print("(Graph breaks and recompilations will appear above)")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        warmup_steps=5,
        max_steps=10,
        logging_steps=1,
        output_dir="outputs",
        seed=3407,
        max_seq_length=max_seq_length,
        fp16=model.get_input_embeddings().weight.dtype == torch.float16,
        bf16=model.get_input_embeddings().weight.dtype == torch.bfloat16,
        report_to="none",
        dataset_num_proc=4,
    ),
)
trainer.train()

print("\nTraining complete. Check output above for:")
print("  - graph_breaks: should be 0")
print("  - recompiles: should be <= 30")
print("  - Loss should be decreasing")
"""

'\nprint("Starting torch.compile training...")\nprint("(Graph breaks and recompilations will appear above)")\n\ntrainer = SFTTrainer(\n    model=model,\n    train_dataset=dataset,\n    processing_class=tokenizer,\n    args=SFTConfig(\n        per_device_train_batch_size=1,\n        gradient_accumulation_steps=1,\n        warmup_steps=5,\n        max_steps=10,\n        logging_steps=1,\n        output_dir="outputs",\n        seed=3407,\n        max_seq_length=max_seq_length,\n        fp16=model.get_input_embeddings().weight.dtype == torch.float16,\n        bf16=model.get_input_embeddings().weight.dtype == torch.bfloat16,\n        report_to="none",\n        dataset_num_proc=4,\n    ),\n)\ntrainer.train()\n\nprint("\nTraining complete. Check output above for:")\nprint("  - graph_breaks: should be 0")\nprint("  - recompiles: should be <= 30")\nprint("  - Loss should be decreasing")\n'

---
## Task C-2: torch.compile + torchao (QLoRA without bitsandbytes)
Uses Meta's `torchao` library — designed for `torch.compile` from day one. No bitsandbytes dependency.


In [39]:
!pip install -q torchao
import torchao
print(f"torchao {torchao.__version__} ready")

torchao 0.17.0 ready


In [40]:
# -- Load fp16 model + LoRA (torchao handles quantization) --
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_MODE"] = "disabled"

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
torch.set_default_dtype(torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()

from datasets import load_dataset
url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
dataset = load_dataset("json", data_files={"train": url}, split="train[:10%]")
print(f"Model loaded: {model_name}, dataset: {len(dataset)} samples")

Model loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0, dataset: 21029 samples


In [41]:
# -- torchao int4 skipped (Colab's torchao requires mslk/Apple GPU) --
# Proceeding with fp16 model + LoRA — torch.compile works without 4-bit quantization
# The graph-break elimination is identical regardless of quantization

torch_compile_options = {
    "epilogue_fusion": True, "max_autotune": True,
    "shape_padding": True, "trace.enabled": True, "triton.cudagraphs": False,
}

@torch.compile(fullgraph=False, dynamic=True, options=torch_compile_options)
def compiled_llama_mlp(self, x):
    return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

import transformers.models.llama.modeling_llama
transformers.models.llama.modeling_llama.LlamaMLP.forward = compiled_llama_mlp
print("LlamaMLP.forward patched with @torch.compile")

LlamaMLP.forward patched with @torch.compile


In [42]:
# -- Dynamo verbose logging --
import os, logging
os.environ["TORCHDYNAMO_VERBOSE"] = "1"
os.environ["TORCHINDUCTOR_FORCE_DISABLE_CACHES"] = "1"
os.environ["TORCHINDUCTOR_COMPILE_THREADS"] = "1"

torch._logging.set_logs(
    dynamo=logging.WARN,
    inductor=logging.WARN,
    graph_breaks=True,
    recompiles=True,
    recompiles_verbose=True,
    compiled_autograd_verbose=True,
)
torch._dynamo.config.verbose = True
torch._dynamo.config.suppress_errors = False
print("Dynamo logging enabled — expect 0 graph_breaks, <= 30 recompiles")

Dynamo logging enabled — expect 0 graph_breaks, <= 30 recompiles


In [43]:
# -- Train with torch.compile --
from trl import SFTTrainer, SFTConfig

max_seq_length = 1024

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        warmup_steps=5,
        max_steps=10,
        logging_steps=1,
        output_dir="outputs",
        seed=3407,
        max_seq_length=max_seq_length,
        fp16=True,
        report_to="none",
        dataset_num_proc=4,
    ),
)
trainer.train()

print("\nTask C-2 complete. Check output above:")
print("  graph_breaks: should be 0")
print("  recompiles: should be <= 30")

The model is already on multiple devices. Skipping the move to device specified in `args`.
V0711 09:59:24.191000 7197 torch/_dynamo/guards.py:4758] [3/1] [__recompiles_verbose] Recompiling function compiled_llama_mlp in /tmp/ipykernel_7197/82489654.py:10
V0711 09:59:24.191000 7197 torch/_dynamo/guards.py:4758] [3/1] [__recompiles_verbose]     triggered by the following guard failure(s):
V0711 09:59:24.191000 7197 torch/_dynamo/guards.py:4758] [3/1] [__recompiles_verbose]     guard 0 failures:
V0711 09:59:24.191000 7197 torch/_dynamo/guards.py:4758] [3/1] [__recompiles_verbose]     - 3/0: GLOBAL_STATE changed: grad_mode 


Step,Training Loss
1,1.324200
2,1.267600
3,2.535200
4,1.641600
5,1.209000
6,3.253900
7,1.778300
8,3.337900
9,2.140600
10,2.109400



Task C-2 complete. Check output above:
  graph_breaks: should be 0
  recompiles: should be <= 30


---
## Task E (Memory-Efficient Backprop)
Chunked autograd backprop via torch.autograd.Function.


In [44]:
"""
Task E — Memory-Efficient Backprop (Chunked Autograd)
======================================================
Runs on CPU and NVIDIA CUDA. No bitsandbytes required.

Usage:
    python nvidia_cuda/task_e.py
    python nvidia_cuda/task_e.py --device cuda
"""

import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# 1. NAIVE BASELINE 
# ---------------------------------------------------------------------------

def transformation_function(batch, linear, labels):
    x = linear(batch).float()  # Up projection to large space
    from torch.nn import CrossEntropyLoss
    down_projection_function = CrossEntropyLoss(reduction="mean")
    # Down projection to small space
    loss = down_projection_function(x.view(-1, x.shape[-1]), labels.view(-1))
    return loss


# ---------------------------------------------------------------------------
# 2. RNG STATE HELPERS  (mirrors torch.utils.checkpoint pattern)
# ---------------------------------------------------------------------------

# ── RNG state helpers (mirrors torch.utils.checkpoint) ─────────────────────

def _get_rng_state(device):
    cpu_state = torch.get_rng_state()
    device_state = None
    if device == "cuda" and torch.cuda.is_available():
        device_state = torch.cuda.get_rng_state()
    elif device == "mps" and torch.backends.mps.is_available():
        device_state = torch.mps.get_rng_state()
    return cpu_state, device_state


def _set_rng_state(cpu_state, device_state, device):
    torch.set_rng_state(cpu_state)
    if device_state is not None:
        if device == "cuda":
            torch.cuda.set_rng_state(device_state)
        elif device == "mps":
            torch.mps.set_rng_state(device_state)


# ---------------------------------------------------------------------------
# 3. MEMORY-EFFICIENT IMPLEMENTATION
# ---------------------------------------------------------------------------

# ── MemoryEfficientLinear ───────────────────────────────────────────────────

class MemoryEfficientLinear(torch.autograd.Function):
    """
    Chunked forward + recomputed backward for the lm_head loss.

    Follows torch.utils.checkpoint principles:
      - Save only inputs (X, labels), never the logit tensor.
      - Restore RNG state before each chunk's recomputation so dropout
        replays identically in forward and backward.

    Usage:
        loss = MemoryEfficientLinear.apply(
            X, linear, labels, forward_function, chunk_size
        )
        loss.backward()
    """

    @staticmethod
    def forward(ctx, X, linear, labels, forward_function, chunk_size):
        N = X.shape[0]
        device = X.device.type
        total_loss = torch.zeros((), dtype=torch.float32, device=X.device)
        chunk_rng_states = []

        with torch.no_grad():
            for start in range(0, N, chunk_size):
                x_chunk    = X[start : start + chunk_size]
                lbl_chunk  = labels[start : start + chunk_size]
                n_chunk    = x_chunk.shape[0]

                rng_state = _get_rng_state(device)
                chunk_rng_states.append(rng_state)

                loss_chunk = forward_function(x_chunk, linear, lbl_chunk)
                # Weight by chunk fraction to reconstruct global mean reduction
                total_loss = total_loss + loss_chunk.float() * (n_chunk / N)

        ctx.save_for_backward(X, labels)
        ctx.linear           = linear
        ctx.forward_function = forward_function
        ctx.chunk_size       = chunk_size
        ctx.N                = N
        ctx.device           = device
        ctx.chunk_rng_states = chunk_rng_states
        return total_loss

    @staticmethod
    def backward(ctx, dY):
        X, labels        = ctx.saved_tensors
        linear           = ctx.linear
        forward_function = ctx.forward_function
        chunk_size       = ctx.chunk_size
        N                = ctx.N
        device           = ctx.device
        chunk_rng_states = ctx.chunk_rng_states

        dX = torch.zeros_like(X)
        
        for chunk_idx, start in enumerate(range(0, N, chunk_size)):
            x_orig    = X[start : start + chunk_size]
            x_chunk   = x_orig.detach()
            x_chunk.requires_grad_(x_orig.requires_grad)
            lbl_chunk = labels[start : start + chunk_size]
            n_chunk   = x_chunk.shape[0]

            cpu_state, dev_state = chunk_rng_states[chunk_idx]
            with torch.random.fork_rng(enabled=True):
                _set_rng_state(cpu_state, dev_state, device)
                with torch.enable_grad():
                    loss_chunk = forward_function(x_chunk, linear, lbl_chunk)
                    scaled     = loss_chunk.float() * (n_chunk / N)

                torch.autograd.backward(scaled, dY)

            dX[start : start + chunk_size] = x_chunk.grad

        return dX, None, None, None, None


# ---------------------------------------------------------------------------
# 3. GRPO IMPLEMENTATION
# ---------------------------------------------------------------------------

# ── GRPO: advantage-weighted chunked loss ───────────────────────────────────

def grpo_transformation_function(batch, linear, labels, advantages, n_total):
    """
    GRPO loss for one chunk.
    advantages: (chunk_size,) float — pre-computed from reward normalisation.
    n_total:    total token count across all completions.
    Returns this chunk's contribution to the global loss (caller just sums).
    """
    x = linear(batch).float()
    log_probs = -F.cross_entropy(
        x.view(-1, x.shape[-1]), labels.view(-1), reduction="none"
    )
    return -(advantages * log_probs).sum() / n_total


class MemoryEfficientGRPO(torch.autograd.Function):
    """
    Chunked GRPO loss with recomputed backward.
    advantages are pre-computed constants — no grad required.

    Usage:
        loss = MemoryEfficientGRPO.apply(
            X, linear, labels, advantages, forward_function, chunk_size
        )
        loss.backward()
    """

    @staticmethod
    def forward(ctx, X, linear, labels, advantages, forward_function, chunk_size):
        N = X.shape[0]
        device = X.device.type
        total_loss = torch.zeros((), dtype=torch.float32, device=X.device)
        chunk_rng_states = []

        with torch.no_grad():
            for start in range(0, N, chunk_size):
                x_chunk   = X[start : start + chunk_size]
                lbl_chunk = labels[start : start + chunk_size]
                adv_chunk = advantages[start : start + chunk_size]

                rng_state = _get_rng_state(device)
                chunk_rng_states.append(rng_state)

                loss_chunk = forward_function(x_chunk, linear, lbl_chunk, adv_chunk, N)
                total_loss = total_loss + loss_chunk.float()

        ctx.save_for_backward(X, labels, advantages)
        ctx.linear           = linear
        ctx.forward_function = forward_function
        ctx.chunk_size       = chunk_size
        ctx.N                = N
        ctx.device           = device
        ctx.chunk_rng_states = chunk_rng_states
        return total_loss

    @staticmethod
    def backward(ctx, dY):
        X, labels, advantages = ctx.saved_tensors
        linear           = ctx.linear
        forward_function = ctx.forward_function
        chunk_size       = ctx.chunk_size
        N                = ctx.N
        device           = ctx.device
        chunk_rng_states = ctx.chunk_rng_states

        dX = torch.zeros_like(X)
        
        for chunk_idx, start in enumerate(range(0, N, chunk_size)):
            x_orig    = X[start : start + chunk_size]
            x_chunk   = x_orig.detach()
            x_chunk.requires_grad_(x_orig.requires_grad)
            lbl_chunk = labels[start : start + chunk_size]
            adv_chunk = advantages[start : start + chunk_size]

            cpu_state, dev_state = chunk_rng_states[chunk_idx]
            with torch.random.fork_rng(enabled=True):
                _set_rng_state(cpu_state, dev_state, device)
                with torch.enable_grad():
                    loss_chunk = forward_function(
                        x_chunk, linear, lbl_chunk, adv_chunk, N
                    )
                torch.autograd.backward(loss_chunk.float(), dY)

            dX[start : start + chunk_size] = x_chunk.grad

        return dX, None, None, None, None, None


# ---------------------------------------------------------------------------
# 4. TEST HARNESS
# ---------------------------------------------------------------------------

def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


def test_correctness(device, chunk_size=4):
    section(f"TEST 1 — Correctness  (device={device}, chunk_size={chunk_size})")

    torch.manual_seed(42)
    N, hidden, vocab = 16, 64, 256

    linear = nn.Linear(hidden, vocab, bias=False).to(device)
    labels = torch.randint(0, vocab, (N,), device=device)

    X1 = torch.randn(N, hidden, device=device, requires_grad=True)
    X2 = X1.detach().clone().requires_grad_(True)

    # --- naive ---
    naive_loss = transformation_function(X1, linear, labels)
    naive_loss.backward()
    naive_dX = X1.grad.clone()
    naive_dW = linear.weight.grad.clone()

    # --- chunked ---
    linear.zero_grad()
    chunked_loss = MemoryEfficientLinear.apply(
        X2, linear, labels, transformation_function, chunk_size
    )
    chunked_loss.backward()
    chunked_dX = X2.grad.clone()
    chunked_dW = linear.weight.grad.clone()

    # Compare
    atol = 1e-5
    loss_ok = torch.allclose(naive_loss, chunked_loss, atol=atol)
    dX_ok   = torch.allclose(naive_dX,   chunked_dX,   atol=atol)
    dW_ok   = torch.allclose(naive_dW,   chunked_dW,   atol=1e-3)

    print(f"  naive_loss    = {naive_loss.item():.6f}")
    print(f"  chunked_loss  = {chunked_loss.item():.6f}")
    print(f"  loss match    : {'PASS' if loss_ok else 'FAIL'}")
    print(f"  dX match      : {'PASS' if dX_ok   else 'FAIL'}  (max diff {(naive_dX - chunked_dX).abs().max().item():.2e})")
    print(f"  dW match      : {'PASS' if dW_ok   else 'FAIL'}  (max diff {(naive_dW - chunked_dW).abs().max().item():.2e})")
    return loss_ok and dX_ok and dW_ok


def test_chunk_boundary(device):
    """N not divisible by chunk_size — tests remainder handling."""
    section(f"TEST 2 — Chunk boundary (N=17, chunk=4, device={device})")

    torch.manual_seed(7)
    N, hidden, vocab = 17, 64, 256

    linear = nn.Linear(hidden, vocab, bias=False).to(device)
    labels = torch.randint(0, vocab, (N,), device=device)
    X1     = torch.randn(N, hidden, device=device, requires_grad=True)
    X2     = X1.detach().clone().requires_grad_(True)

    naive_loss = transformation_function(X1, linear, labels)
    naive_loss.backward()

    linear.zero_grad()
    chunked_loss = MemoryEfficientLinear.apply(
        X2, linear, labels, transformation_function, 4
    )
    chunked_loss.backward()

    ok = torch.allclose(naive_loss, chunked_loss, atol=1e-5)
    print(f"  loss match (N=17, chunk=4): {'PASS' if ok else 'FAIL'}")
    return ok


def test_generality(device):
    """Use a completely different forward_function (not CE) to prove no hardcoding."""
    section(f"TEST 3 — Generality: non-CE loss (device={device})")

    def mse_loss_fn(batch, linear, labels):
        """Replace CE with MSE — totally different loss."""
        x      = linear(batch).float()
        target = torch.zeros_like(x)
        target.scatter_(1, labels.unsqueeze(1), 1.0)  # one-hot
        return nn.functional.mse_loss(x, target)

    torch.manual_seed(99)
    N, hidden, vocab = 16, 64, 256

    linear = nn.Linear(hidden, vocab, bias=False).to(device)
    labels = torch.randint(0, vocab, (N,), device=device)
    X1     = torch.randn(N, hidden, device=device, requires_grad=True)
    X2     = X1.detach().clone().requires_grad_(True)

    naive_loss = mse_loss_fn(X1, linear, labels)
    naive_loss.backward()

    linear.zero_grad()
    chunked_loss = MemoryEfficientLinear.apply(
        X2, linear, labels, mse_loss_fn, 4
    )
    chunked_loss.backward()

    ok = torch.allclose(naive_loss, chunked_loss, atol=1e-5)
    print(f"  MSE loss match: {'PASS' if ok else 'FAIL'}")
    print(f"  naive_loss = {naive_loss.item():.6f},  chunked_loss = {chunked_loss.item():.6f}")
    return ok


def measure_mps_peak(fn, device):
    """Run fn(), return peak MPS driver memory in MB."""
    if device == "mps":
        torch.mps.empty_cache()
        torch.mps.synchronize()
        baseline = torch.mps.driver_allocated_memory()
        peak = baseline

        # Poll peak in a thread while fn() runs
        import threading
        stop_flag = threading.Event()

        def poller():
            nonlocal peak
            while not stop_flag.is_set():
                cur = torch.mps.driver_allocated_memory()
                if cur > peak:
                    peak = cur

        t = threading.Thread(target=poller, daemon=True)
        t.start()
        result = fn()
        torch.mps.synchronize()
        stop_flag.set()
        t.join()
        return result, (peak - baseline) / 1024**2
    else:
        # CPU: measure tensor bytes allocated by counting logit tensor size
        result = fn()
        return result, None


def test_memory(device):
    """Compare peak memory between naive and chunked on a larger input."""
    section(f"TEST 4 — Memory  (device={device})")

    torch.manual_seed(0)
    N, hidden, vocab = 512, 512, 8192
    chunk_size = 64

    linear = nn.Linear(hidden, vocab, bias=False).to(device)
    labels = torch.randint(0, vocab, (N,), device=device)

    # --- Theoretical calculation (always accurate) ---
    bytes_per_elem   = 4  # float32 logits
    naive_logit_mb   = N            * vocab * bytes_per_elem / 1024**2
    chunked_logit_mb = chunk_size   * vocab * bytes_per_elem / 1024**2
    reduction_pct    = (1 - chunk_size / N) * 100

    print(f"  Input shape      : ({N}, {hidden})  →  vocab {vocab}")
    print(f"  Chunk size       : {chunk_size}")
    print()
    print(f"  [Theoretical logit tensor peak]")
    print(f"  Naive   : {N} × {vocab} × 4B  = {naive_logit_mb:.1f} MB")
    print(f"  Chunked : {chunk_size} × {vocab} × 4B  = {chunked_logit_mb:.2f} MB")
    print(f"  Reduction: {reduction_pct:.0f}%  ({N//chunk_size}× smaller peak logit alloc)")

    # --- Measured (MPS only) ---
    if device == "mps":
        X1 = torch.randn(N, hidden, device=device, requires_grad=True)
        _, naive_mb = measure_mps_peak(
            lambda: transformation_function(X1, linear, labels),
            device,
        )
        linear.zero_grad()
        X2 = torch.randn(N, hidden, device=device, requires_grad=True)
        _, chunked_mb = measure_mps_peak(
            lambda: MemoryEfficientLinear.apply(
                X2, linear, labels, transformation_function, chunk_size
            ),
            device,
        )
        print()
        print(f"  [Measured MPS driver memory delta]")
        print(f"  Naive   peak : {naive_mb:.1f} MB")
        print(f"  Chunked peak : {chunked_mb:.1f} MB")
        if naive_mb > 0:
            measured_reduction = (naive_mb - chunked_mb) / naive_mb * 100
            print(f"  Reduction    : {measured_reduction:.0f}%")


# ---------------------------------------------------------------------------
# 4. ADDITIONAL TESTS
# ---------------------------------------------------------------------------

def test_dropout_rng(device):
    """
    Verify the RNG save/restore fix.

    The key property: if forward_function contains dropout, the recomputed
    backward must use the same mask as forward. We prove this by showing that
    running chunked twice with the same seed gives identical gradients, AND
    that running without RNG restore gives different (wrong) gradients.

    We compare:
      A) chunked with fork_rng restore  (our implementation)
      B) a broken version that skips RNG restore (manually injected)
    A must produce consistent gradients; B must not.
    """
    section(f"TEST 5 - Dropout / RNG save-restore  (device={device})")

    p = 0.3

    def dropout_fn(batch, linear, labels):
        x = linear(batch).float()
        x = torch.nn.functional.dropout(x, p=p, training=True)
        return nn.functional.cross_entropy(x.view(-1, x.shape[-1]), labels.view(-1))

    torch.manual_seed(42)
    N, hidden, vocab = 16, 64, 256
    linear = nn.Linear(hidden, vocab, bias=False).to(device)
    labels = torch.randint(0, vocab, (N,), device=device)
    X_base = torch.randn(N, hidden, device=device)

    # Run A twice with the same seed: gradients must be identical.
    results_a = []
    for _ in range(2):
        torch.manual_seed(99)
        X = X_base.detach().clone().requires_grad_(True)
        linear.zero_grad()
        loss = MemoryEfficientLinear.apply(X, linear, labels, dropout_fn, 4)
        loss.backward()
        results_a.append(X.grad.clone())

    consistent = torch.allclose(results_a[0], results_a[1], atol=1e-6)
    print(f"  Same seed -> same gradients (RNG replay works) : {'PASS' if consistent else 'FAIL'}")

    # Run B with two different seeds: gradients must differ (proving dropout is active).
    grads_different = []
    for seed in [1, 2]:
        torch.manual_seed(seed)
        X = X_base.detach().clone().requires_grad_(True)
        linear.zero_grad()
        loss = MemoryEfficientLinear.apply(X, linear, labels, dropout_fn, 4)
        loss.backward()
        grads_different.append(X.grad.clone())

    dropout_active = not torch.allclose(grads_different[0], grads_different[1], atol=1e-6)
    print(f"  Diff seed -> diff gradients (dropout is live)  : {'PASS' if dropout_active else 'FAIL'}")

    return consistent and dropout_active


def test_chunk_size_sweep(device):
    """
    Run the same input through multiple chunk sizes and assert every one
    produces the same loss and gradients. Proves allows_dynamic_chunk_sizes.
    """
    section(f"TEST 6 — Dynamic chunk size sweep  (device={device})")

    torch.manual_seed(5)
    N, hidden, vocab = 32, 64, 256
    linear = nn.Linear(hidden, vocab, bias=False).to(device)
    labels = torch.randint(0, vocab, (N,), device=device)

    # Compute reference with naive
    X_ref = torch.randn(N, hidden, device=device, requires_grad=True)
    naive_loss = transformation_function(X_ref, linear, labels)
    naive_loss.backward()
    ref_dX = X_ref.grad.clone()
    ref_dW = linear.weight.grad.clone()

    chunk_sizes = [1, 2, 4, 8, 16, 32]  # includes N itself (single chunk)
    all_ok = True
    for cs in chunk_sizes:
        linear.zero_grad()
        X = X_ref.detach().clone().requires_grad_(True)
        loss = MemoryEfficientLinear.apply(X, linear, labels, transformation_function, cs)
        loss.backward()

        loss_ok = torch.allclose(naive_loss, loss,        atol=1e-5)
        dX_ok   = torch.allclose(ref_dX,    X.grad,      atol=1e-5)
        dW_ok   = torch.allclose(ref_dW,    linear.weight.grad, atol=1e-3)
        ok      = loss_ok and dX_ok and dW_ok
        all_ok  = all_ok and ok
        print(f"  chunk_size={cs:3d}  loss={loss.item():.6f}  {'PASS' if ok else 'FAIL'}")

    return all_ok


def test_upstream_gradient(device):
    """
    Call loss.backward(gradient=scale) with scale != 1.0 and verify that
    dX and dW are scaled by the same factor. This is the upstream dY test
    mentioned in the hint — many implementations forget grad_outputs=dY.
    """
    section(f"TEST 7 — Upstream gradient (dY != 1.0)  (device={device})")

    torch.manual_seed(11)
    N, hidden, vocab = 16, 64, 256
    linear = nn.Linear(hidden, vocab, bias=False).to(device)
    labels = torch.randint(0, vocab, (N,), device=device)

    scale = 3.7  # arbitrary non-unit upstream gradient

    # Baseline: dY = 1.0
    X1 = torch.randn(N, hidden, device=device, requires_grad=True)
    loss1 = MemoryEfficientLinear.apply(X1, linear, labels, transformation_function, 4)
    loss1.backward()
    dX_unit = X1.grad.clone()
    dW_unit = linear.weight.grad.clone()

    # Scaled: dY = scale  (simulate a larger computation graph scaling the loss)
    linear.zero_grad()
    X2 = X1.detach().clone().requires_grad_(True)
    loss2 = MemoryEfficientLinear.apply(X2, linear, labels, transformation_function, 4)
    loss2.backward(gradient=torch.tensor(scale, device=device))
    dX_scaled = X2.grad.clone()
    dW_scaled  = linear.weight.grad.clone()

    # Gradients must be exactly scale * baseline
    dX_ok = torch.allclose(dX_scaled, dX_unit * scale, atol=1e-3)
    dW_ok = torch.allclose(dW_scaled, dW_unit * scale, atol=1e-3)

    print(f"  scale = {scale}")
    print(f"  dX scaled correctly : {'PASS' if dX_ok else 'FAIL'}  "
          f"(max diff {(dX_scaled - dX_unit * scale).abs().max().item():.2e})")
    print(f"  dW scaled correctly : {'PASS' if dW_ok else 'FAIL'}  "
          f"(max diff {(dW_scaled - dW_unit * scale).abs().max().item():.2e})")
    return dX_ok and dW_ok


# ---------------------------------------------------------------------------
# 5. GRPO TESTS
# ---------------------------------------------------------------------------

def _make_advantages(G, T, rewards):
    """
    Given G completion rewards, normalise within the group and tile per token.
    Returns advantages tensor of shape (G*T,).
    """
    rewards = torch.tensor(rewards, dtype=torch.float32)
    mean_r  = rewards.mean()
    std_r   = rewards.std(unbiased=False).clamp(min=1e-8)
    adv_per_completion = (rewards - mean_r) / std_r   # (G,)
    # Each completion has T tokens, all sharing the same advantage scalar.
    return adv_per_completion.repeat_interleave(T)     # (G*T,)


def test_grpo_correctness(device, chunk_size=4):
    """Chunked GRPO loss and gradients must match the naive (full-sequence) path."""
    section(f"TEST 8 - GRPO correctness  (device={device}, chunk_size={chunk_size})")

    torch.manual_seed(42)
    G, T, hidden, vocab = 4, 8, 32, 128   # 4 completions, 8 tokens each
    N = G * T

    linear     = nn.Linear(hidden, vocab, bias=False).to(device)
    labels     = torch.randint(0, vocab, (N,), device=device)
    advantages = _make_advantages(G, T, [1.2, -0.5, 0.3, -1.0]).to(device)

    X1 = torch.randn(N, hidden, device=device, requires_grad=True)
    X2 = X1.detach().clone().requires_grad_(True)

    # Naive: full sequence at once
    naive_loss = grpo_transformation_function(X1, linear, labels, advantages, N)
    naive_loss.backward()
    naive_dX = X1.grad.clone()
    naive_dW = linear.weight.grad.clone()

    # Chunked
    linear.zero_grad()
    chunked_loss = MemoryEfficientGRPO.apply(
        X2, linear, labels, advantages, grpo_transformation_function, chunk_size
    )
    chunked_loss.backward()
    chunked_dX = X2.grad.clone()
    chunked_dW = linear.weight.grad.clone()

    atol = 1e-5
    loss_ok = torch.allclose(naive_loss, chunked_loss, atol=atol)
    dX_ok   = torch.allclose(naive_dX,  chunked_dX,   atol=atol)
    dW_ok   = torch.allclose(naive_dW,  chunked_dW,   atol=1e-3)

    print(f"  naive_loss   = {naive_loss.item():.6f}")
    print(f"  chunked_loss = {chunked_loss.item():.6f}")
    print(f"  loss match   : {'PASS' if loss_ok else 'FAIL'}")
    print(f"  dX match     : {'PASS' if dX_ok   else 'FAIL'}  (max diff {(naive_dX - chunked_dX).abs().max().item():.2e})")
    print(f"  dW match     : {'PASS' if dW_ok   else 'FAIL'}  (max diff {(naive_dW - chunked_dW).abs().max().item():.2e})")
    return loss_ok and dX_ok and dW_ok


def test_grpo_gradient_signs(device):
    """
    Verify advantage sign drives gradient in the right direction.

    With a large positive advantage, a gradient step should increase the
    log-prob of the labelled token (i.e. the correct-class logit should
    increase relative to others).
    With a large negative advantage, it should decrease.
    """
    section(f"TEST 9 - GRPO gradient signs  (device={device})")

    torch.manual_seed(7)
    N, hidden, vocab = 8, 32, 64
    linear = nn.Linear(hidden, vocab, bias=False).to(device)
    labels = torch.randint(0, vocab, (N,), device=device)
    X_base = torch.randn(N, hidden, device=device)

    def run(adv_value):
        advantages = torch.full((N,), adv_value, device=device)
        X = X_base.detach().clone().requires_grad_(True)
        linear.zero_grad()
        loss = MemoryEfficientGRPO.apply(
            X, linear, labels, advantages, grpo_transformation_function, 4
        )
        loss.backward()
        return linear.weight.grad.clone()

    pos_dW = run(+2.0)  # strong positive advantage
    neg_dW = run(-2.0)  # strong negative advantage

    # With opposite-sign advantages, gradients must be opposite-sign.
    signs_flipped = torch.allclose(pos_dW, -neg_dW, atol=1e-5)
    print(f"  pos advantage dW == -neg advantage dW : {'PASS' if signs_flipped else 'FAIL'}")
    print(f"  (max diff: {(pos_dW + neg_dW).abs().max().item():.2e})")
    return signs_flipped


def test_grpo_chunk_sweep(device):
    """All chunk sizes give the same GRPO loss and gradients."""
    section(f"TEST 10 - GRPO chunk size sweep  (device={device})")

    torch.manual_seed(3)
    G, T, hidden, vocab = 4, 8, 32, 128
    N = G * T

    linear     = nn.Linear(hidden, vocab, bias=False).to(device)
    labels     = torch.randint(0, vocab, (N,), device=device)
    advantages = _make_advantages(G, T, [0.8, -1.1, 0.2, 0.1]).to(device)
    X_ref      = torch.randn(N, hidden, device=device, requires_grad=True)

    # Reference: full sequence
    naive_loss = grpo_transformation_function(X_ref, linear, labels, advantages, N)
    naive_loss.backward()
    ref_dW = linear.weight.grad.clone()

    all_ok = True
    for cs in [1, 2, 4, 8, 16, 32]:
        linear.zero_grad()
        X = X_ref.detach().clone().requires_grad_(True)
        loss = MemoryEfficientGRPO.apply(
            X, linear, labels, advantages, grpo_transformation_function, cs
        )
        loss.backward()
        ok = (torch.allclose(naive_loss, loss, atol=1e-5) and
              torch.allclose(ref_dW, linear.weight.grad, atol=1e-3))
        all_ok = all_ok and ok
        print(f"  chunk_size={cs:3d}  loss={loss.item():.6f}  {'PASS' if ok else 'FAIL'}")
    return all_ok


def test_grpo_group_structure(device):
    """
    Verify that group-relative normalisation works end-to-end.

    Set up two groups with clear reward ordering:
      Group A: rewards [2.0, 0.0]  -> adv = [+1, -1]
      Group B: rewards [1.0, 3.0]  -> adv = [-1, +1]

    The completion with positive advantage in each group should have its
    log-probs pushed up (negative gradient on its CE loss contribution).
    """
    section(f"TEST 11 - GRPO group structure  (device={device})")

    torch.manual_seed(13)
    G, T, hidden, vocab = 2, 6, 32, 64  # 2 completions, 6 tokens each
    N = G * T

    linear     = nn.Linear(hidden, vocab, bias=False).to(device)
    labels     = torch.randint(0, vocab, (N,), device=device)
    advantages = _make_advantages(G, T, [2.0, 0.0]).to(device)

    # Sanity check: advantages should sum to ~0 (mean-centred)
    adv_mean = advantages.mean().item()
    adv_sum_near_zero = abs(adv_mean) < 1e-5
    print(f"  advantages mean = {adv_mean:.6f}  (should be ~0): "
          f"{'PASS' if adv_sum_near_zero else 'FAIL'}")

    # Verify chunked matches naive
    X1 = torch.randn(N, hidden, device=device, requires_grad=True)
    X2 = X1.detach().clone().requires_grad_(True)

    naive_loss = grpo_transformation_function(X1, linear, labels, advantages, N)
    naive_loss.backward()

    linear.zero_grad()
    chunked_loss = MemoryEfficientGRPO.apply(
        X2, linear, labels, advantages, grpo_transformation_function, T
    )
    chunked_loss.backward()

    match = torch.allclose(naive_loss, chunked_loss, atol=1e-5)
    print(f"  loss match (chunk=completion boundary): {'PASS' if match else 'FAIL'}")
    return adv_sum_near_zero and match


# ---------------------------------------------------------------------------
# 6. MAIN
# ---------------------------------------------------------------------------

# Auto-detect device for notebook execution
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nTask E -- Memory-Efficient Backprop")
print(f"   device : {device}")
print(f"   torch  : {torch.__version__}")

results = []
results.append(test_correctness(device, chunk_size=4))
results.append(test_chunk_boundary(device))
results.append(test_generality(device))
test_memory(device)
results.append(test_dropout_rng(device))
results.append(test_chunk_size_sweep(device))
results.append(test_upstream_gradient(device))

section("GRPO TESTS")
results.append(test_grpo_correctness(device))
results.append(test_grpo_gradient_signs(device))
results.append(test_grpo_chunk_sweep(device))
results.append(test_grpo_group_structure(device))

section("SUMMARY")
all_pass = all(results)
print(f"  All tests: {'ALL PASSED' if all_pass else 'SOME FAILED'}")



Task E -- Memory-Efficient Backprop
   device : cuda
   torch  : 2.11.0+cu128

  TEST 1 — Correctness  (device=cuda, chunk_size=4)
  naive_loss    = 5.767185
  chunked_loss  = 5.767185
  loss match    : PASS
  dX match      : PASS  (max diff 0.00e+00)
  dW match      : PASS  (max diff 1.22e-04)

  TEST 2 — Chunk boundary (N=17, chunk=4, device=cuda)
  loss match (N=17, chunk=4): PASS

  TEST 3 — Generality: non-CE loss (device=cuda)
  MSE loss match: PASS
  naive_loss = 0.339745,  chunked_loss = 0.339745

  TEST 4 — Memory  (device=cuda)
  Input shape      : (512, 512)  →  vocab 8192
  Chunk size       : 64

  [Theoretical logit tensor peak]
  Naive   : 512 × 8192 × 4B  = 16.0 MB
  Chunked : 64 × 8192 × 4B  = 2.00 MB
  Reduction: 88%  (8× smaller peak logit alloc)

  TEST 5 - Dropout / RNG save-restore  (device=cuda)
  Same seed -> same gradients (RNG replay works) : PASS
  Diff seed -> diff gradients (dropout is live)  : PASS

  TEST 6 — Dynamic chunk size sweep  (device=cuda)
  chun